# 서울시 외식업 원자료 재수집과 버전 검증
종로구·마포구, 외식업 10종, 2023~2025년 분석에 필요한 서울시 원자료를 다시 수집한다. 공식 파일은 서울 전역·전체 업종 단위로 내려받고 분석 표본 선택은 08번에서 한다.

기본 실행은 **네트워크를 사용하지 않고 마지막 수집 기록을 검증**한다. 실제 수집은 커널 시작 전에 `SEOUL_COLLECT=1`을 설정한 경우에만 실행된다. 기존 `data/raw/` 파일을 덮어쓰지 않고 날짜별 새 하위 폴더를 만든다. 접근 차단·HTTP 오류가 발생하면 재시도 없이 해당 회차 전체를 중단한다.

공식 출처: [추정매출](https://data.seoul.go.kr/dataList/OA-15572/S/1/datasetView.do), [점포](https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do), [상권 경계](https://data.seoul.go.kr/dataList/OA-15560/S/1/datasetView.do).
매출 페이지에는 2024년부터 공간 단위가 표준단위구역으로 변경되었다는 안내가 있다. 현재 경계와 코드가 일치해도 과거 경계의 일관성이 입증되는 것은 아니다.


이 노트북은 필요할 때만 사용하는 **수집 관리 도구**이며 최종 분석의 전체 실행에는 포함하지 않는다. 수집 및 스냅샷 등록은 공용 원자료와 `metadata/`에 기록된다. `final/metadata/`에 고정한 분석 입력은 자동 교체하지 않는다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from html.parser import HTMLParser
import os, json, hashlib, io, zipfile, urllib.request, urllib.parse, urllib.error, uuid, time, re
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'AGENTS.md').exists())
META = ROOT/'metadata'
REPORT = ROOT/'reports'
LATEST = META/'seoul_refresh_latest.json'
COLLECT = os.environ.get('SEOUL_COLLECT','0') == '1'
YEARS = [2023,2024,2025]
FOOD_CODES = {f'CS100{i:03d}' for i in range(1,11)}
SOURCES = {'sales':'OA-15572','stores':'OA-15577','areas':'OA-15560'}
ENDPOINT = 'https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?&useCache=false'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def relative(path): return Path(path).relative_to(ROOT).as_posix()
def now(): return datetime.now(timezone.utc).isoformat()
print('수집 실행' if COLLECT else '오프라인 기록 검증')


## 1. 공식 페이지에서 현재 파일 번호 확인
페이지의 실제 다운로드 항목을 파싱해 연도별 파일 번호를 선택한다. 이전 코드의 파일 번호를 검증 없이 재사용하지 않는다. 파일명·수정일 안내의 원문은 페이지 HTML로 함께 보존한다.


In [ ]:
class DownloadItems(HTMLParser):
    def __init__(self):
        super().__init__(); self.items=[]
    def handle_starttag(self, tag, attrs):
        a=dict(attrs)
        action=a.get('onclick','')+' '+a.get('href','')
        m=re.search(r"downloadFile\('([0-9]+)'\)",action)
        title=a.get('title','')
        if m and title.lower().endswith('.zip'):
            row={'title':title,'seq':m.group(1)}
            if row not in self.items: self.items.append(row)

class CollectionStopped(Exception):
    pass

def request_bytes(url, data=None):
    request=urllib.request.Request(url,data=data,headers={
        'User-Agent':'Mozilla/5.0','Referer':'https://data.seoul.go.kr/'})
    try:
        with urllib.request.urlopen(request,timeout=60) as response:
            body=response.read()
            status=response.status
    except urllib.error.HTTPError as exc:
        raise CollectionStopped(f'HTTP {exc.code}; 재시도하지 않음') from None
    except (urllib.error.URLError,TimeoutError):
        raise CollectionStopped('연결 오류 또는 시간 초과; 재시도하지 않음') from None
    head=body[:8192].decode('utf-8',errors='ignore').lower()
    if any(marker in head for marker in ['request blocked','access denied','access blocked']):
        raise CollectionStopped('접근 차단 응답; 재시도하지 않음')
    return body, status

def save_new(path, content):
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('xb') as f: f.write(content)

def persist_run(manifest, run_dir):
    text=json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False)
    target=run_dir/'manifest.json'
    # 회차 진행 기록만 갱신하며 원본 ZIP·CSV·페이지 파일은 수정하지 않는다.
    meta_run=META/'seoul_recollection'/manifest['run_id']
    meta_run.mkdir(parents=True,exist_ok=True)
    (meta_run/'manifest.json').write_text(text,encoding='utf-8')
    LATEST.write_text(text,encoding='utf-8')


## 2. 6개 연도 파일과 현재 경계 수집
ZIP 무결성을 확인하고 원본 바이트와 해시를 기록한다. 압축 내부 경로를 그대로 추출하지 않고 허용한 CSV·공간 파일을 새 폴더에 저장한다. 공개 CSV와 기존 CSV의 바이트가 같으면 수집 시점은 새로워도 데이터 내용은 동일하다고 기록한다.


In [ ]:
manifest = json.loads(LATEST.read_text(encoding='utf-8')) if LATEST.exists() else None
if COLLECT:
    run_id=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
    run_dir=ROOT/'data/raw/seoul_recollection'/run_id
    run_dir.mkdir(parents=True,exist_ok=False)
    originals=[ROOT/f'data/raw/seoul_{kind}/{kind}_{year}.csv'
               for kind in ['sales','stores'] for year in YEARS]
    originals += list((ROOT/'data/raw/seoul_areas/extracted').glob('*'))
    original_hashes={relative(p):sha(p) for p in originals if p.is_file()}
    manifest={'run_id':run_id,'started_at':now(),'status':'running','items':[],
              'source_pages':[],'original_sha256':original_hashes,
              'scope':{'districts':['종로구','마포구'],'years':YEARS,'industry_codes':sorted(FOOD_CODES)},
              'boundary_note':'현재 경계; 2024년 표준단위구역 변경 안내. 과거 경계 불변을 의미하지 않음.'}
    persist_run(manifest,run_dir)
    try:
        for kind,inf_id in SOURCES.items():
            page_url=f'https://data.seoul.go.kr/dataList/{inf_id}/S/1/datasetView.do'
            body,status=request_bytes(page_url)
            page_path=run_dir/'source_pages'/f'{kind}.html'
            save_new(page_path,body)
            parser=DownloadItems(); parser.feed(body.decode('utf-8'))
            assert parser.items, f'{kind} 다운로드 항목 없음'
            manifest['source_pages'].append({'dataset':kind,'url':page_url,
                'path':relative(page_path),'sha256':sha(page_path),'http_status':status})
            selected=[]
            for year in YEARS if kind!='areas' else [None]:
                candidates=[i for i in parser.items if (str(year)+'년' in i['title'])] if year else parser.items
                assert len(candidates)==1, f'{kind} {year} 다운로드 항목 모호'
                selected.append((year,candidates[0]))
            for year,item in selected:
                time.sleep(1)
                params={'infId':inf_id,'seq':item['seq'],'seqNo':'','infSeq':'3'}
                blob,status=request_bytes(ENDPOINT,urllib.parse.urlencode(params).encode())
                archive=zipfile.ZipFile(io.BytesIO(blob))
                assert archive.testzip() is None, 'ZIP 무결성 실패'
                prefix=f'{kind}_{year}' if year else 'areas'
                zip_path=run_dir/kind/(prefix+'.zip')
                save_new(zip_path,blob)
                members=[]
                eligible=[i for i in archive.infolist() if not i.is_dir()]
                if kind!='areas':
                    csv_members=[i for i in eligible if i.filename.lower().endswith('.csv')]
                    assert len(csv_members)==1, 'CSV 수가 1이 아님'
                    eligible=csv_members
                for member in eligible:
                    ext=Path(member.filename).suffix.lower()
                    if kind=='areas' and ext not in {'.shp','.shx','.dbf','.prj','.cpg'}: continue
                    path=run_dir/kind/(prefix+ext)
                    save_new(path,archive.read(member))
                    old=(ROOT/f'data/raw/seoul_{kind}/{prefix}.csv') if kind!='areas' else next(
                        (ROOT/'data/raw/seoul_areas/extracted').glob('*'+ext),None)
                    old_hash=sha(old) if old and old.exists() else None
                    members.append({'path':relative(path),'archive_member':member.filename,
                        'bytes':path.stat().st_size,'sha256':sha(path),
                        'previous_path':relative(old) if old and old.exists() else None,
                        'previous_sha256':old_hash,'same_as_previous':sha(path)==old_hash})
                if kind=='areas':
                    assert {Path(m['path']).suffix for m in members}>={'.shp','.shx','.dbf','.prj'}
                manifest['items'].append({'dataset':kind,'year':year,'source':page_url,
                    'file_sequence':item['seq'],'file_title':item['title'],'request_parameters':params,
                    'path':relative(zip_path),'bytes':len(blob),'sha256':sha(zip_path),
                    'recorded_at':now(),'http_status':status,'extracted':members,
                    'same_as_previous':all(m['same_as_previous'] for m in members)})
                persist_run(manifest,run_dir)
                print(kind,year or '현재 경계','수집·ZIP 검증 완료',flush=True)
        manifest['status']='downloaded'
    except Exception as exc:
        manifest['status']='stopped'
        manifest['error_type']=type(exc).__name__
        manifest['error']=str(exc)[:240]
        manifest['stopped_without_retry']=True
    finally:
        manifest['finished_at']=now()
        manifest['original_files_unchanged']=all(sha(ROOT/p)==h for p,h in original_hashes.items())
        persist_run(manifest,run_dir)
assert manifest is not None, '수집 기록이 없습니다. 수집 전용 커널에서 SEOUL_COLLECT=1로 실행하세요.'
print('회차:',manifest['run_id'],'상태:',manifest['status'],'확보 파일:',len(manifest['items']))


## 3. 연도·업종·키와 파일 해시 검증
확보된 6개 파일의 외식업 10종, 4개 분기, 키 중복을 확인한다. 2025년 점포 파일의 영문 열 이름은 검증용 표준 이름으로만 바꾸며 저장 원본을 수정하지 않는다. 미제공 매출은 채우지 않는다.


In [ ]:
checks=[]
profiles=[]
for item in manifest['items']:
    zip_ok=sha(ROOT/item['path'])==item['sha256']
    checks.append({'check':f"{item['dataset']}_{item['year']}_zip_hash",'passed':zip_ok})
    for member in item['extracted']:
        checks.append({'check':member['path']+'_hash','passed':sha(ROOT/member['path'])==member['sha256']})
    if item['dataset']=='areas': continue
    csv_member=next(m for m in item['extracted'] if m['path'].endswith('.csv'))
    columns=pd.read_csv(ROOT/csv_member['path'],encoding='cp949',nrows=0).columns
    rename={'기준_년분기_코드':'quarter','상권_코드':'area_id','서비스_업종_코드':'industry_id',
            'stdr_yyqu_cd':'quarter','trdar_cd':'area_id','svc_induty_cd':'industry_id'}
    cols=[c for c in columns if c in rename]
    frame=pd.read_csv(ROOT/csv_member['path'],encoding='cp949',usecols=cols,dtype=str).rename(columns=rename)
    assert {'quarter','area_id','industry_id'}<=set(frame)
    food=frame.loc[frame['industry_id'].isin(FOOD_CODES)]
    quarters=set(frame['quarter'])
    expected={str(item['year'])+str(q) for q in range(1,5)}
    check={'all_food_codes':set(food['industry_id'])==FOOD_CODES,
           'expected_four_quarters':quarters==expected,
           'unique_food_key':not food.duplicated(['quarter','area_id','industry_id']).any()}
    checks.extend({'check':f"{item['dataset']}_{item['year']}_{name}",'passed':bool(passed)}
                  for name,passed in check.items())
    profiles.append({'dataset':item['dataset'],'year':item['year'],'all_rows':len(frame),
        'food_rows':len(food),'food_codes':food['industry_id'].nunique(),
        'same_as_previous':csv_member['same_as_previous'],'csv_path':csv_member['path']})
check_frame=pd.DataFrame(checks)
profile_frame=pd.DataFrame(profiles)
display(profile_frame)
display(check_frame.groupby('passed').size().rename('검증 수').to_frame())


## 4. 분석 입력으로 채택할 수집본
6개 연도 파일과 경계가 모두 확보되고 검증을 통과해야 `validated` 상태가 된다. 중단된 회차가 있으면 마지막으로 검증된 입력 매니페스트는 유지한다. 분석 노트북은 `metadata/seoul_analysis_sources.json`만 읽고 네트워크 요청을 하지 않는다.


In [ ]:
complete = len(profiles)==6 and len(manifest['items'])==7
passed = complete and bool(check_frame['passed'].all()) and manifest.get('original_files_unchanged',False)
if COLLECT and passed:
    manifest['status']='validated'
    manifest['validation_passed']=True
    manifest['validated_at']=now()
    persist_run(manifest,ROOT/'data/raw/seoul_recollection'/manifest['run_id'])
    (META/'seoul_analysis_sources.json').write_text(
        json.dumps(manifest,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
check_frame.to_csv(REPORT/'11_seoul_refresh_checks.csv',index=False,encoding='utf-8-sig')
profile_frame.to_csv(REPORT/'11_seoul_refresh_profiles.csv',index=False,encoding='utf-8-sig')
summary={'run_id':manifest['run_id'],'status':manifest['status'],'all_required_files_available':complete,
         'checks_passed':int(check_frame['passed'].sum()),'checks_total':len(check_frame),
         'csv_contents_unchanged':bool(profile_frame['same_as_previous'].all()) if complete else None,
         'validated_input_manifest':'metadata/seoul_analysis_sources.json' if passed else None}
(REPORT/'11_seoul_refresh_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(Markdown('**검증된 새 수집본을 분석 입력으로 등록했습니다.**' if passed else
                 '**이번 수집본은 완전한 입력으로 등록하지 않았습니다. 이전 검증본을 유지합니다.**'))
display(pd.DataFrame([summary]))


원본 내용이 같으면 기존 분석 결과가 유지되는 것이 정상이다. 새 좌표·공식 ID 등 KOPIS 수집본의 변화는 별도의 03→04→08→09→10→06→07 실행에서 반영한다. 서울시 경계 파일은 현재 제공본이므로 2023~2025년 경계 불변이나 제공기준 불변을 입증하지 않는다.
